In [ ]:
!pip install -q transformers accelerate torch torchvision pillow


# Load BLIP-2 Model

In [ ]:
import torch
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Blip2Processor.from_pretrained(
    "Salesforce/blip2-flan-t5-xl"
)

model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl",
    torch_dtype=torch.float16,
    device_map="auto"
)


# Upload a Cropped Damage Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from PIL import Image

image_path = "/content/drive/MyDrive/damage_crops/dent_01.jpg"  # CHANGE THIS
image = Image.open(image_path).convert("RGB")
image


# Define the EXACT Prompt Template

In [ ]:
prompt = """
You are an automotive damage assessment assistant.

You are given:
• A cropped image of a vehicle damage
• The detected damage type from an object detection model
• The damage area ratio relative to the full image

Your task:
1. Assess the severity of the damage.
2. Decide the required repair action.

Severity must be one of:
- Minor
- Moderate
- Severe

Repair action must be one of:
Dent:
- Dent pull only
- Dent pull + repaint
- Panel repair or replacement

Scratch:
- Polish only
- Repaint
- Fill and repaint

Damage type: dent
Area ratio: 0.037

Respond ONLY in JSON format:
{
  "severity": "<Minor | Moderate | Severe>",
  "repair_action": "<one valid action>",
  "confidence": "<0.0 – 1.0>"
}
"""


# Run VLM Inference

In [ ]:
inputs = processor(
    images=image,
    text=prompt,
    return_tensors="pt"
).to(device, torch.float16)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=200
)

output = processor.decode(
    generated_ids[0],
    skip_special_tokens=True
)

print(output)


# Validate Output

In [ ]:
{
  "severity": "Moderate",
  "repair_action": "Dent pull + repaint",
  "confidence": 0.78
}
